# Validate EXP-009: Projection Postprocess — Sanity Check

**🖥 RUNS ON: Kaggle CPU — no GPU needed**

| Item | Value |
|------|-------|
| Target env | Kaggle CPU (no accelerator) |
| Compute | < 2 min |
| Purpose | Verify projection logic on synthetic well data before running on real data |

Uses **synthetic** wells — no competition data required.  
Run this first to confirm the projection implementation is correct.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

print('Kaggle CPU validation — EXP-009 Projection')
print('No competition data required.')

In [ ]:
# ── Projection implementation (inline) ───────────────────────────
def _robfit(s, u, deg=5, n_iter=4, c=2.0):
    w = np.ones(len(s))
    u_fit = u.copy()
    for _ in range(n_iter):
        coeffs = np.polyfit(s, u, deg=deg, w=w)
        u_fit  = np.polyval(coeffs, s)
        resid  = u - u_fit
        mad    = np.median(np.abs(resid - np.median(resid)))
        scale  = 1.4826 * mad if mad > 1e-12 else 1.0
        w      = (np.abs(resid) / scale <= c).astype(float)
        w      = np.maximum(w, 1e-6)
    return u_fit

def apply_projection(df, pred_col='tvt_pred', well_col='well_code',
                     md_col='MD', z_col='Z', tvt_col='TVT',
                     is_eval_col='is_eval', deg=5, n_iter=4, c=2.0):
    corrected = df[pred_col].copy().astype(float)
    n_done = 0
    for _, wdf in df.groupby(well_col):
        wdf   = wdf.sort_values(md_col)
        known = wdf[~wdf[is_eval_col].astype(bool)]
        eval_ = wdf[wdf[is_eval_col].astype(bool)]
        if len(known) == 0 or len(eval_) < deg + 1:
            continue
        last   = known.iloc[-1]
        anchor = float(last[tvt_col]) + float(last[z_col])
        span   = float(eval_[md_col].iloc[-1]) - float(last[md_col])
        if span <= 0:
            continue
        s     = (eval_[md_col].values - float(last[md_col])) / span
        u     = eval_[pred_col].values + eval_[z_col].values - anchor
        u_fit = _robfit(s, u, deg=deg, n_iter=n_iter, c=c)
        corrected.loc[eval_.index] = u_fit - eval_[z_col].values + anchor
        n_done += 1
    return corrected, n_done

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print('Functions defined.')

In [ ]:
# ── Generate synthetic wells ──────────────────────────────────────
def make_synthetic_wells(n_wells=5, n_known=20, n_eval=30, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for w in range(n_wells):
        md_known = np.linspace(0, 2000, n_known)
        md_eval  = np.linspace(2010, 3000, n_eval)
        z_known  = -md_known * 0.6 + rng.normal(0, 5, n_known)
        z_eval   = -md_eval  * 0.6 + rng.normal(0, 5, n_eval)
        tvt_known      = -z_known * 0.5 + rng.normal(0, 3, n_known)
        tvt_true_eval  = -z_eval  * 0.5 + rng.normal(0, 3, n_eval)
        tvt_pred_eval  = tvt_true_eval + rng.normal(0, 8, n_eval)  # imperfect model
        for md, z, tvt in zip(md_known, z_known, tvt_known):
            rows.append({'well_code': f'W{w}', 'MD': md, 'Z': z, 'TVT': tvt,
                         'tvt_pred': tvt, 'is_eval': False})
        for md, z, tvt_t, tvt_p in zip(md_eval, z_eval, tvt_true_eval, tvt_pred_eval):
            rows.append({'well_code': f'W{w}', 'MD': md, 'Z': z, 'TVT': tvt_t,
                         'tvt_pred': tvt_p, 'is_eval': True})
    return pd.DataFrame(rows)

df = make_synthetic_wells(n_wells=5, n_known=20, n_eval=30)
print(f'Synthetic data: {df.shape}  wells: {df["well_code"].nunique()}')
df.head(3)

In [ ]:
# ── Run projection and evaluate ───────────────────────────────────
eval_mask = df['is_eval'].astype(bool)
rmse_before = rmse(df.loc[eval_mask, 'TVT'].values, df.loc[eval_mask, 'tvt_pred'].values)

corrected, n_done = apply_projection(df, deg=5, n_iter=4, c=2.0)

rmse_after = rmse(df.loc[eval_mask, 'TVT'].values, corrected[eval_mask].values)

print(f'Wells processed: {n_done}')
print(f'RMSE before: {rmse_before:.4f}')
print(f'RMSE after:  {rmse_after:.4f}')
print(f'Delta:       {rmse_after - rmse_before:+.4f}')

assert rmse_after < rmse_before, 'FAIL: projection did not improve RMSE on synthetic data'
print('\nPASS: projection improves predictions — safe to run exp009_projection_postprocess__CPU')

In [ ]:
# ── Per-well breakdown ────────────────────────────────────────────
print('Per-well RMSE:')
for w, wdf in df[eval_mask].groupby('well_code'):
    idx = wdf.index
    rb  = rmse(df.loc[idx, 'TVT'].values, df.loc[idx, 'tvt_pred'].values)
    ra  = rmse(df.loc[idx, 'TVT'].values, corrected.loc[idx].values)
    print(f'  {w}: before={rb:.3f}  after={ra:.3f}  delta={ra-rb:+.3f}')